In [22]:
import numpy as np
import random
import math
import pandas as pd


In [23]:

df = pd.read_csv("../../data/agaricus-lepiota.data", header=None)

In [24]:
x=df.iloc[:,1:]
y=df.iloc[:,0]
df

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,e,k,s,n,f,n,a,c,b,y,...,s,o,o,p,o,o,p,b,c,l
8120,e,x,s,n,f,n,a,c,b,y,...,s,o,o,p,n,o,p,b,v,l
8121,e,f,s,n,f,n,a,c,b,n,...,s,o,o,p,o,o,p,b,c,l
8122,p,k,y,n,f,y,f,c,n,b,...,k,w,w,p,w,o,e,w,v,l


In [25]:
def boostrap_sample(x,y,seed=0):
    data = pd.concat([y, x], axis=1)

    random.seed(a=seed)
    newdf=[]
    
    for i in range(len(data)-1):
        newdf.append(data.iloc[random.randint(0,len(data)-1)])
    df=pd.DataFrame(newdf)
    x=df.iloc[:,1:]
    y=df.iloc[:,0]
    return x,y
    
boostrap_sample(x,y,42)


(     1  2  3  4  5  6  7  8  9  10  ... 13 14 15 16 17 18 19 20 21 22
 5238  k  f  n  f  n  f  w  n  w  e  ...  f  w  n  p  w  o  e  w  v  l
 912   x  s  w  t  l  f  c  b  g  e  ...  s  w  w  p  w  o  p  k  n  g
 204   f  y  n  t  l  f  c  b  n  e  ...  y  w  w  p  w  o  p  k  y  p
 6074  x  y  e  f  s  f  c  n  b  t  ...  k  w  p  p  w  o  e  w  v  l
 2253  f  f  g  t  n  f  c  b  p  t  ...  s  p  w  p  w  o  p  n  v  d
 ...  .. .. .. .. .. .. .. .. .. ..  ... .. .. .. .. .. .. .. .. .. ..
 1595  f  f  g  f  n  f  w  b  k  t  ...  f  w  w  p  w  o  e  n  a  g
 275   x  y  n  t  a  f  c  b  w  e  ...  y  w  w  p  w  o  p  n  s  p
 6422  x  s  e  f  f  f  c  n  b  t  ...  k  w  w  p  w  o  e  w  v  l
 4911  f  s  e  t  n  f  c  b  w  e  ...  s  w  e  p  w  t  e  w  c  w
 4182  x  y  y  f  f  f  c  b  g  e  ...  k  p  n  p  w  o  l  h  y  d
 
 [8123 rows x 22 columns],
 5238    e
 912     e
 204     e
 6074    p
 2253    e
        ..
 1595    e
 275     e
 6422    p
 4911    e
 4182    

In [26]:
def calculate_entropy(y):
    total_rows = len(y)
    target_values = y.unique()
    entropy = 0
    for value in target_values:
        # Calculate the proportion of instances with the current value
        value_count = len(y[y == value])
        proportion = value_count / total_rows
        entropy-=proportion*math.log2(proportion)
    return entropy

def calculate_information_gain(x,y,feature):
    # Calculate weighted average entropy for the feature
    unique_values = x[feature].unique()
    weighted_entropy = 0
    for value in unique_values:
        subset = x[x[feature] == value]
        subset_y = y[x[feature] == value]

        proportion = len(subset) / len(x)
        weighted_entropy += proportion * calculate_entropy(subset_y)

    # Calculate information gain
    information_gain = calculate_entropy(y) - weighted_entropy

    return information_gain

#https://www.geeksforgeeks.org/machine-learning/sklearn-iterative-dichotomiser-3-id3-algorithms/
#TQM geeksforgeeks
def build_id3_tree(x,y):
    features=x.columns
    if len(y.unique()) == 1:
        return y.iloc[0]
    if len(features) == 0:
        return y.mode().iloc[0]

    best_feature = max(features,key=lambda feature: calculate_information_gain(x, y, feature))
    tree = {best_feature: {}}

    features = [f for f in features if f != best_feature]
    for value in x[best_feature].unique():
        subset = x[x[best_feature] == value]
        subset_y = y[x[best_feature] == value]
        tree[best_feature][value] = id3(subset[features],subset_y)

    return tree


build_id3_tree(x,y)

{5: {'p': 'p',
  'a': 'e',
  'l': 'e',
  'n': {20: {'n': 'e',
    'k': 'e',
    'w': {22: {'w': 'e',
      'l': {3: {'c': 'e', 'n': 'e', 'w': 'p', 'y': 'p'}},
      'd': {8: {'n': 'p', 'b': 'e'}},
      'g': 'e',
      'p': 'e'}},
    'h': 'e',
    'r': 'p',
    'o': 'e',
    'y': 'e',
    'b': 'e'}},
  'f': 'p',
  'c': 'p',
  'y': 'p',
  's': 'p',
  'm': 'p'}}

In [35]:
def build_random_forest(x,y,n_trees=10, random_state=0):
    random.seed(a=random_state)
    arboles=[]
    for i in range(n_trees):
        x1,y2=boostrap_sample(x,y)
        arboles.append(build_id3_tree(x1,y2))
    return arboles
build_random_forest(x,y)

[{5: {'s': 'p',
   'n': {20: {'k': 'e',
     'w': {22: {'d': {8: {'b': 'e', 'n': 'p'}},
       'g': 'e',
       'l': {3: {'c': 'e', 'n': 'e', 'w': 'p', 'y': 'p'}},
       'p': 'e',
       'w': 'e'}},
     'n': 'e',
     'b': 'e',
     'h': 'e',
     'r': 'p',
     'o': 'e',
     'y': 'e'}},
   'y': 'p',
   'l': 'e',
   'f': 'p',
   'c': 'p',
   'p': 'p',
   'a': 'e',
   'm': 'p'}},
 {5: {'s': 'p',
   'n': {20: {'k': 'e',
     'w': {22: {'d': {8: {'b': 'e', 'n': 'p'}},
       'g': 'e',
       'l': {3: {'c': 'e', 'n': 'e', 'w': 'p', 'y': 'p'}},
       'p': 'e',
       'w': 'e'}},
     'n': 'e',
     'b': 'e',
     'h': 'e',
     'r': 'p',
     'o': 'e',
     'y': 'e'}},
   'y': 'p',
   'l': 'e',
   'f': 'p',
   'c': 'p',
   'p': 'p',
   'a': 'e',
   'm': 'p'}},
 {5: {'s': 'p',
   'n': {20: {'k': 'e',
     'w': {22: {'d': {8: {'b': 'e', 'n': 'p'}},
       'g': 'e',
       'l': {3: {'c': 'e', 'n': 'e', 'w': 'p', 'y': 'p'}},
       'p': 'e',
       'w': 'e'}},
     'n': 'e',
     'b': 'e',


In [36]:
def predict_tree(tree, row):
    while isinstance(tree, dict):
        feature = next(iter(tree))
        value = row[feature]
        tree = tree[feature][value]

    return tree


def predict_ensemble(trees, X):
    predictions = []
    for tree in trees:
        tree_predictions = []

        for row in X.to_numpy():
            tree_predictions.append(predict_tree(tree, row))

        predictions.append(tree_predictions)

    predictions = np.array(predictions)

    final_predictions = []

    for i in range(X.shape[0]):
        classes, counts = np.unique(predictions[:, i], return_counts=True)
        max_count = counts.max()

        final_predictions.append(classes[counts == max_count][0])

    return np.array(final_predictions)


predict_ensemble(build_random_forest(x,y),x.iloc[1:10])

array(['p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p'], dtype='<U1')